<a href="https://colab.research.google.com/github/komazawa-deep-learning/komazawa-deep-learning.github.io/blob/master/2026notebooks/2026_1023Image_captioning_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 画像キャプション生成 (Flickr8k 日本語) 実習

* author: 浅川 伸一 educ0233@komazawa-u.ac.jp
* date: 2026_1023
* note:

Google Colab で実行する場合の注意：
* 以下の共有ドライブフォルダをマウントする。
`/content/drive/Shareddrives/2026人工知能_浅川/Image_aptioning`
* この共有フォルダには，以下のファイルが含まれている：
    * `captions-ja.csv`: 日本語キャプション(Flickr8k の機械翻訳版)
    * `images`: サンプル画像の一部
    *  `feats_resnet18_7x7.npy(.json)`: ResNet18 の画像特徴(計算済み)
    *  `captioner_best.pt`: 学習済みモデル
    *  `tokenizer/`:                   モデルと対になるトークナイザ
    * `config.json`:   設定・画像の分割・配布時の得点など

* 注意：画像データの使用については Flickr8K データセットの制作者が権利を保有している。
そのため，授業目的でのみ使用を認める。外部への持ち出し・再配布は禁止とする。

* 出典
    * 元データ: Flickr8k (Rashtchian, Young, Hodosh, Hockenmaier, NAACL HLT 2010 Workshop)
    * 日本語訳: Kaggle tanreinama/flicker8k-japanese-translation(機械翻訳)

In [ ]:
# 0. 設定
SHARE_NAME = "2026人工知能_浅川/Image_captioning"   # 共有された Drive のフォルダ名
TRAIN      = False    # False: 配布された学習済みモデルを使う / True: 自分で再学習する(演習用)

# 以下は TRAIN = True のときに効く(False のときは、配布された設定が使われる)
SEED        = 42
PRETOK      = "script"   # 自作 BPE の前処理。"script": 文字種の境目で先に区切る / "metaspace": 文全体を 1 単位にする
VOCAB_SIZE  = 8000       # 自作 BPE の語彙数
EMB, HID, ATT, DROP = 256, 512, 256, 0.3
BATCH       = 128
EPOCHS      = 20
PATIENCE    = 4
LR          = 4e-4
ALPHA_C     = 0.0        # 注意の二重確率的正則化の重み(演習 1)
MAX_TOK     = 50

In [ ]:
# 1. 必要なライブラリ(無ければ入れる)
import importlib, subprocess, sys

def ensure(pip_name, module_name=None):
    try:
        importlib.import_module(module_name or pip_name)
    except ImportError:
        print(f"{pip_name} をインストールします...")
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name])
        if r.returncode != 0:
            print(f"  [警告] {pip_name} のインストールに失敗しました")

for p in ["tokenizers", "transformers", "sacrebleu", "pycocoevalcap"]:
    ensure(p)

import os, re, json, time, math, unicodedata, random
from collections import Counter
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# 2. Drive の共有フォルダを探し、必要なファイルを手元(/content/work)にコピーする
import glob, shutil, zipfile
DRIVE_ROOT = "/content/drive"
WORK = f"/content/drive/Shareddrives/{SHARE_NAME}" if os.path.isdir("/content") else "./work"
os.makedirs(WORK, exist_ok=True)

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount(DRIVE_ROOT)

def find_share(name, root=DRIVE_ROOT):
    pats = [f"/content/drive/Shareddrives/{name}/images"]                       # マイドライブに置いたとき
    # pats = [f"{root}/MyDrive/{name}",                       # マイドライブに置いたとき
    #         f"{root}/.shortcut-targets-by-id/*/{name}",     # 共有されたフォルダにショートカットを作ったとき
    #         f"{root}/Shareddrives/*/{name}"]                # 共有ドライブのとき
    for p in pats:
        hit = glob.glob(p)
        if hit: return hit[0]
    return None

SHARE = find_share(SHARE_NAME)
if SHARE is None:
    mine = sorted(os.listdir(f"{DRIVE_ROOT}/MyDrive"))[:30] if os.path.isdir(f"{DRIVE_ROOT}/MyDrive") else []
    raise FileNotFoundError(f"共有フォルダ '{SHARE_NAME}' が見つかりません。\n"
                            "・フォルダにショートカットを追加しましたか(「最初に 1 回だけ」の手順 2)\n"
                            "・フォルダ名は先生の指示と一致していますか\n"
                            f"マイドライブの中身(先頭 30): {mine}")
print("共有フォルダ:", SHARE)
#print(open(f"{SHARE}/README.txt", encoding="utf-8").read())

#for f in ["captions-ja.csv", "feats_resnet18_7x7.npy", "feats_resnet18_7x7.npy.json", "captioner_best.pt"]:
for f in ["captions-ja.csv", "feats_resnet18_7x7.npy", "feats_resnet18_7x7.npy.json", "captioner_best.pt", "config.json"]:
    if not os.path.exists(f"{WORK}/{f}"):
        shutil.copy(f"{SHARE}/{f}", f"{WORK}/{f}"); print("コピー:", f)
if not os.path.isdir(f"{WORK}/tokenizer"):
    shutil.copytree(f"{SHARE}/tokenizer", f"{WORK}/tokenizer"); print("コピー: tokenizer/")
IMG_DIR = f"{WORK}/images"
# if not os.path.isdir(IMG_DIR) or len(os.listdir(IMG_DIR)) < 100:
#     print("画像を展開しています(数分かかることがある)...")
#     zpath = f"{WORK}/flickr8k_images.zip" q
#     shutil.copy(f"{SHARE}/flickr8k_images.zip", zpath)
#     with zipfile.ZipFile(zpath) as z: z.extractall(IMG_DIR)
#     os.remove(zpath)
print("画像:", len(os.listdir(IMG_DIR)), "枚")
captions_file = f"{WORK}/captions-ja.csv"

In [ ]:
# 3. 配布時の設定を読み込む
cfg = json.load(open(f"{WORK}/config.json", encoding="utf-8"))
print("配布時の試験得点:", cfg["metrics_test"])
print("配布時の環境:", cfg["versions"])
if not TRAIN:                                   # 学習済みモデルを使うときは、配布時の設定に合わせる
    for k in ["SEED", "PRETOK", "VOCAB_SIZE", "EMB", "HID", "ATT", "DROP", "MAX_TOK"]:
        globals()[k] = cfg[k]
        assert cfg.get("NLAYERS", 1) == 1, "このノートの Captioner は LSTM 1 層です"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# 4. キャプションの読み込み
def find_file(name, roots=("/content", ".", "/kaggle/input")):
    for root in roots:
        if not os.path.isdir(root): continue
        for dp, _, fs in os.walk(root):
            if name in fs: return os.path.join(dp, name)
    return None

try:
    captions_file
except NameError:
    captions_file = find_file("captions-ja.csv")
print("captions_file:", captions_file)

df = pd.read_csv(captions_file)                       # 先頭行が列名(header=0 が既定)
cols = df.columns.tolist(); print("列名:", cols)
IMG_COL = "image" if "image" in cols else cols[0]
CAP_COL = next((c for c in cols if c.replace("_", "-").lower() in ("caption-ja", "captions-ja")), cols[-1])
print(f"画像列={IMG_COL!r}  日本語キャプション列={CAP_COL!r}")
df = df[[IMG_COL, CAP_COL]].dropna(); df.columns = ["image", "text"]
df["text"] = df["text"].astype(str).str.strip(); df = df[df["text"] != ""]
by_image = df.groupby("image", sort=True)["text"].apply(list).to_dict()
images = sorted(by_image)
print(f"キャプション {len(df):,} 文 / 画像 {len(images):,} 枚 / 1 枚あたり {dict(Counter(len(v) for v in by_image.values()))}")

In [ ]:
df

In [ ]:
df[df['image'] == '1287982439_6578006e22.jpg']

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import torchvision, torchvision.transforms as T
view = T.Compose([T.Resize(256), T.CenterCrop(224)])               # 特徴抽出と同じ切り出し

img_fname = '/content/drive/Shareddrives/2026人工知能_浅川/Image_captioning/images/1287982439_6578006e22.jpg'
img = view(Image.open(img_fname).convert("RGB"))
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
# 5. 画像ファイルの場所と、配布時の分割
def find_images(names, roots=(WORK,)):
    want, found, seen = set(names), {}, set()
    for root in roots:
        if not os.path.isdir(root) or os.path.realpath(root) in seen: continue
        seen.add(os.path.realpath(root))
        for dp, _, fs in os.walk(root):
            for f in fs:
                if f in want and f not in found and f.lower().endswith((".jpg", ".jpeg", ".png")):
                    found[f] = os.path.join(dp, f)
    return found

found = find_images(images)
print(f"画像ファイル {len(found):,} / {len(images):,} 件が見つかった")

In [ ]:
type(cfg), cfg.keys(), type(cfg['te_imgs']), len(cfg['te_imgs'])
cfg['V'],
found

In [ ]:
# 6. 自作 BPE トークナイザ
from tokenizers import Tokenizer, Regex
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Metaspace, Split, Sequence
from transformers import PreTrainedTokenizerFast

def train_bpe(texts, vocab_size, pretok):
    raw = Tokenizer(BPE(unk_token="[UNK]"))
    try:
        meta = Metaspace(replacement="▁", prepend_scheme="always")
    except TypeError:
        meta = Metaspace(replacement="▁", add_prefix_space=True)
    if pretok == "script":
        script = Split(Regex(r"[一-鿿々〇]+|[぀-ゟ]+|[゠-ヿー]+|[A-Za-z]+|[0-9]+|."), behavior="isolated")
        raw.pre_tokenizer = Sequence([script, meta])
    else:
        raw.pre_tokenizer = meta
    trainer = BpeTrainer(vocab_size=vocab_size, min_frequency=2,
                         special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]"])
    raw.train_from_iterator(texts, trainer=trainer)
    return PreTrainedTokenizerFast(tokenizer_object=raw, unk_token="[UNK]", pad_token="[PAD]",
                                   cls_token="[CLS]", sep_token="[SEP]", mask_token="[MASK]")

def caps(imgs):
    return [t for im in imgs for t in by_image[im]]

if TRAIN:
    tok = train_bpe(caps(tr_imgs), VOCAB_SIZE, PRETOK)
else:
    tok = PreTrainedTokenizerFast.from_pretrained(f"{WORK}/tokenizer")      # 配布されたトークナイザ(モデルと対)
V = len(tok)
UNK, PAD, BOS, EOS = tok.unk_token_id, tok.pad_token_id, tok.cls_token_id, tok.sep_token_id   # 文頭は [CLS]、文末は [SEP] を流用
print(f"語彙数 {V}  (UNK={UNK}, PAD={PAD}, 文頭={BOS}, 文末={EOS})")

# for t in random.sample(caps(tr_imgs), 3):
#     ids = tok(t, add_special_tokens=False)["input_ids"]
#     print(f"{t}\n   {len(ids)} トークン: {tok.convert_ids_to_tokens(ids)}")

# ho = caps(te_imgs)
# ids_ho = tok(ho, add_special_tokens=False)["input_ids"]
# n_unk = sum(i == UNK for s in ids_ho for i in s)
# print(f"\n試験用の文: 1 文あたり {np.mean([len(s) for s in ids_ho]):.1f} トークン、"
#       f"{sum(map(len, ho)) / sum(map(len, ids_ho)):.2f} 文字/トークン、[UNK] {n_unk} 個(学習側に無い文字)")

In [ ]:
s = '明日は午後から雨が降るらしいよ'
print(tok(s))
ids = tok(s)['input_ids']
print(tok.convert_ids_to_tokens(ids))

In [ ]:
# 7. 画像特徴
import torchvision, torchvision.transforms as T
from PIL import Image

def build_backbone(pretrained=True):
    w = torchvision.models.ResNet18_Weights.DEFAULT if pretrained else None
    net = torchvision.models.resnet18(weights=w)
    body = nn.Sequential(*list(net.children())[:-2]).eval()        # 平均プーリングと全結合を除く
    tf = w.transforms() if w is not None else T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])
    return body, tf

class ImgDS(torch.utils.data.Dataset):
    def __init__(self, paths, tf): self.paths, self.tf = paths, tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(Image.open(self.paths[i]).convert("RGB"))

@torch.no_grad()
def extract_features(paths, pretrained=True, bs=64):
    body, tf = build_backbone(pretrained); body = body.to(device)
    # macOS / Windows のノートブックでは、子プロセス方式(spawn)が __main__ のクラスを読めずに失敗する。Linux(Colab)だけ並列にする
    nw = 2 if sys.platform.startswith("linux") else 0
    dl = torch.utils.data.DataLoader(ImgDS(paths, tf), batch_size=bs, num_workers=nw)
    out = []
    for k, x in enumerate(dl):
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
            f = body(x.to(device))                                  # (B, 512, 7, 7)
        out.append(f.flatten(2).transpose(1, 2).half().cpu())        # (B, 49, 512)
        if k % 20 == 0: print(f"  特徴抽出 {k * bs}/{len(paths)}")
    return torch.cat(out).numpy()

cache = f"{WORK}/feats_resnet18_7x7.npy"
if os.path.exists(cache) and os.path.exists(cache + ".json") and json.load(open(cache + ".json")) == images:
    feats = np.load(cache); print("保存済みの特徴を読み込んだ")
else:
    t0 = time.time()
    feats = extract_features([found[im] for im in images])
    np.save(cache, feats)
    json.dump(images, open(cache + ".json", "w"))
    print(f"特徴抽出 {time.time() - t0:.0f} 秒")
print("特徴:", feats.shape, feats.dtype)
fi = {im: i for i, im in enumerate(images)}
ft = torch.from_numpy(feats).to(device)          # GPU に載せる(fp16)
R, FD = feats.shape[1], feats.shape[2]

In [ ]:
# 8. モデル
class Captioner(nn.Module):
    def __init__(self, V, fdim=FD, emb=EMB, hid=HID, adim=ATT, drop=DROP):
        super().__init__()
        self.fnorm = nn.LayerNorm(fdim)
        self.emb = nn.Embedding(V, emb, padding_idx=PAD)
        self.wf, self.wh, self.v = nn.Linear(fdim, adim), nn.Linear(hid, adim), nn.Linear(adim, 1)   # 注意
        self.init_h, self.init_c = nn.Linear(fdim, hid), nn.Linear(fdim, hid)
        self.beta = nn.Linear(hid, fdim)                       # 文脈ベクトルのゲート
        self.cell = nn.LSTMCell(emb + fdim, hid)
        self.drop = nn.Dropout(drop); self.out = nn.Linear(hid, V)

    def start(self, f):
        f = self.fnorm(f.float()); m = f.mean(1)
        return f, self.wf(f), torch.tanh(self.init_h(m)), torch.tanh(self.init_c(m))

    def step(self, f, pf, y, h, c):
        e = self.v(torch.tanh(pf + self.wh(h)[:, None])).squeeze(-1)       # (B, 領域数)
        a = F.softmax(e, -1)                                               # 注意の重み
        ctx = torch.sigmoid(self.beta(h)) * (a[:, :, None] * f).sum(1)     # 見ている領域の特徴の加重和
        h, c = self.cell(torch.cat([self.emb(y), ctx], -1), (h, c))
        return self.out(self.drop(h)), h, c, a

    def forward(self, f, ys):                                              # ys: (B, T) 文頭 … 文末
        f, pf, h, c = self.start(f)
        lg, al = [], []
        for t in range(ys.size(1) - 1):
            o, h, c, a = self.step(f, pf, ys[:, t], h, c)
            lg.append(o); al.append(a)
        return torch.stack(lg, 1), torch.stack(al, 1)

    @torch.no_grad()
    def generate(self, f, maxlen=MAX_TOK):                                 # 貪欲法
        self.eval(); f, pf, h, c = self.start(f)
        y = torch.full((f.size(0),), BOS, device=f.device)
        done = torch.zeros(f.size(0), dtype=torch.bool, device=f.device)
        ids, als = [], []
        for _ in range(maxlen - 1):
            o, h, c, a = self.step(f, pf, y, h, c)
            o[:, [PAD, BOS, UNK]] = -1e9
            y = o.argmax(-1); y[done] = PAD
            ids.append(y); als.append(a); done |= y == EOS
            if done.all(): break
        ids = torch.stack(ids, 1).tolist(); als = torch.stack(als, 1).cpu().numpy()
        out = []
        for row in ids:
            row = row[:row.index(EOS)] if EOS in row else row
            out.append([i for i in row if i != PAD])
        return out, als

model = Captioner(V).to(device)
print(f"パラメータ数 {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")
if not TRAIN:
    model.load_state_dict(torch.load(f"{WORK}/captioner_best.pt", map_location=device)); model.eval()
    print("配布された学習済みモデルを読み込んだ")

In [ ]:
model

In [ ]:
# 9. 試験画像でキャプションを生成して評価する
def norm(s): return re.sub(r"[\s▁]+", "", unicodedata.normalize("NFKC", s))

def decode(ids): return norm(tok.decode(ids, skip_special_tokens=True))

def generate_all(feat_rows, bs=256):
    hyps, alphas = [], []
    for i in range(0, len(feat_rows), bs):
        ids, al = model.generate(feat_rows[i:i + bs])
        hyps += [decode(x) for x in ids]; alphas += list(al)
    return hyps, alphas

def score(hyps, refs_list):
    """文字単位の BLEU と CIDEr(MeCab など分かち書きに依存しない)。hyps: 画像ごとの 1 文、refs_list: 画像ごとの参照文リスト"""
    import sacrebleu
    H = [norm(h) or "。" for h in hyps]
    Rf = [[norm(r) for r in rs] for rs in refs_list]
    n = min(len(r) for r in Rf); Rf = [r[:n] for r in Rf]
    bleu = sacrebleu.corpus_bleu(H, [[Rf[i][k] for i in range(len(Rf))] for k in range(n)], tokenize="char").score
    try:
        from pycocoevalcap.cider.cider import Cider
        sp = lambda s: " ".join(s)
        cider = 100 * Cider().compute_score({i: [sp(r) for r in Rf[i]] for i in range(len(Rf))},
                                            {i: [sp(H[i])] for i in range(len(Rf))})[0]
    except Exception as e:
        cider = float("nan")
    return {"BLEU(char)": round(bleu, 2), "CIDEr(char)": round(cider, 2)}


In [ ]:
try:
    import japanize_matplotlib
except ImportError:
    !pip install japanize_matplotlib
    import japanize_matplotlib


In [ ]:
found[list(found.keys())[3]]
k=0
im = found[list(found.keys())[k]]
#found[k]
im
#len(feats), type(feats)
#fi[im.strip('/')[-1]]
#feats[fi[im.split('/')[-1]]]

In [ ]:
def Encoder(pretrained=True):
    w = torchvision.models.ResNet18_Weights.DEFAULT if pretrained else None
    net = torchvision.models.resnet18(weights=w)
    body = nn.Sequential(*list(net.children())[:-2]).eval()        # 平均プーリングと全結合を除く
    transform = w.transforms() if w is not None else T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])
    return body, transform

encoder, transform = Encoder()
def encode(img_fname, encoder=encoder, transform=transform):
    img = Image.open(img_fname).convert("RGB")
    x = transform(img).to(device).unsqueeze(0)
    feature = encoder(x)
    out = feature.flatten(2).transpose(1, 2).half().cpu() #.squeeze(0).detach() #.numpy()
    #out = feature.flatten(2).transpose(1, 2).half().cpu().squeeze(0).detach().numpy()
    return out

_env = encode(im)
ids, al = model.generate(_env)
ids, al = ids[0], al[0]
pieces = [norm(p) or "·" for p in tok.convert_ids_to_tokens(ids)] + ["<eos>"]
print(pieces)

In [ ]:
!wget https://imgcdn.cna.com.tw/Jpn/WebJpnPhotos/800/2025/20250314/1051x1500_771062058253.jpg -O img.jpg

In [ ]:
_env = encode('img.jpg')
ids, al = model.generate(_env)
ids, al = ids[0], al[0]
pieces = [norm(p) or "·" for p in tok.convert_ids_to_tokens(ids)] + ["<eos>"]
print(pieces)

img = Image.open('img.jpg').convert("RGB")
plt.imshow(img)

In [ ]:
# 11. 注意の可視化: 各トークンを出すとき、画像のどこを見ていたか

view = T.Compose([T.Resize(256), T.CenterCrop(224)])               # 特徴抽出と同じ切り出し

def show_attention(k, max_tok=9):
    im = found[list(found.keys())[k]]
    img = view(Image.open(im).convert("RGB"))
    #im = te_imgs[k]
    #img = view(Image.open(found[im]).convert("RGB"))
    ids, al = model.generate(te_feat[k:k + 1])
    ids, al = ids[0], al[0]
    pieces = [norm(p) or "·" for p in tok.convert_ids_to_tokens(ids)] + ["<eos>"]
    n = min(len(pieces), max_tok, len(al))
    fig, axs = plt.subplots(1, n + 1, figsize=(1.9 * (n + 1), 2.3))
    axs[0].imshow(img); axs[0].set_title("image", fontsize=9)
    side = int(round(math.sqrt(R)))
    for j in range(n):
        a = al[j].reshape(side, side)
        a = np.array(Image.fromarray((a / a.max() * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)) / 255.0
        axs[j + 1].imshow(img); axs[j + 1].imshow(a, cmap="jet", alpha=0.45)
        axs[j + 1].set_title(pieces[j] if JA_FONT else f"t{j + 1}", fontsize=10)
    for a_ in axs: a_.axis("off")
    fig.suptitle(decode(ids), fontsize=11); fig.tight_layout(rect=[0, 0, 1, 0.9]); plt.show()

for k in range(4):
    show_attention(k)

In [ ]:

te_feat = ft[torch.tensor([fi[im] for im in te_imgs], device=device)]
hyps, te_alphas = generate_all(te_feat)
refs4 = [by_image[im][1:] for im in te_imgs]                      # 1 番目の参照文を除いた 4 文を正解にする
res = {"モデル": score(hyps, refs4),
       "人間(参照の 1 文目)": score([by_image[im][0] for im in te_imgs], refs4)}   # 人間が書いた 1 文の得点 = 上限の目安
print(pd.DataFrame(res).T.to_string())
print(f"生成文の平均長 {np.mean([len(h) for h in hyps]):.1f} 文字 / 参照文 {np.mean([len(r) for rs in refs4 for r in rs]):.1f} 文字")
print(f"異なり文の数 {len(set(hyps))} / {len(hyps)}  (小さいと、どの画像にも同じ文を出している)")

print()
for im, h in list(zip(te_imgs, hyps))[:10]:
    print(f"[{im}]\n  参照: {by_image[im][0]}\n  生成: {h}")